# 00 — Smoke Test

Minimal end-to-end run to verify the cleaned pipeline wires together. Trims epochs, batch size, and top-N for ~3–5 minute completion on Mac MPS or T4. **Run this first** after `pixi install` to confirm everything imports, decodes, scores, trains, and saves before launching real training.

What this notebook tests:
1. `lisardd` package imports without error
2. HierVAE checkpoint loads and decodes
3. MGraphDTA checkpoint loads and scores
4. One short PPO training loop completes and writes to `runs/smoke_run/`
5. Saved artifacts can be loaded back

If any cell fails, the failure points to the broken contract.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available:", torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False)

In [ ]:
from lisardd.config import ExperimentConfig
from lisardd.runner import run_experiment

cfg = ExperimentConfig(
    run_name="smoke_run",
    algo="ppo",
    reward="binding_affinity",
    target="jnk3",
    mode="smoke",
    seed=42,
    device="cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"),
)
print(cfg)

In [ ]:
run_dir = run_experiment(cfg)
print(f"Run saved to: {run_dir}")

In [ ]:
from lisardd.io import load_run

art = load_run(run_dir, load_state=True)
print("history keys:", list(art.history.keys()))
print("top100 rows:", len(art.top100))
print("top reward range:", min(art.top100['reward']), "->", max(art.top100['reward']))
art.top100.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(art.history['average_obj_scores'], label='avg reward')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title(f"Smoke run: {cfg.run_name}")
plt.legend(); plt.grid(True)
plt.show()

## Done

If the run completed and you see a top100 with non-`None` SMILES plus a reward trajectory, the pipeline is wired correctly. Smoke mode does not produce paper-quality results; it only proves the wiring works.